In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from preprocessing import clean,build_preprocessor,add_engineer
import matplotlib.pyplot as plt
SEED = 42

In [2]:
df_raw = pd.read_csv(r"../data/student-mat.csv",sep=";")
df_raw.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [3]:
# df_after_fe = clean_and_engineer(df_raw)
# df_before_fe = df_after_fe.drop(columns=['study_per_absence', 'failure_impact'])

In [4]:
df_after_fe = clean(df_raw)
df_after_fe = add_engineer(df_raw)
if 'study_per_absence' and 'failure_impact' in df_after_fe:
    df_before_fe = df_after_fe.drop(columns=['study_per_absence', 'failure_impact'])
    # df_before_fe = df_after_fe.drop(columns=['total_edu','total_alc'])
    # df_before_fe = df_after_fe.drop(columns=['G2_adjusted'])
else: 
    df_before_fe=df_after_fe
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=SEED),
    "Gradient Boosting": GradientBoostingRegressor(random_state=SEED)
}

def evaluate_models(df):
    X = df.drop('G3',axis=1)
    y = df['G3']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    results = {}
    for name, model in models.items():
        pipe = Pipeline([
            ('preprocessor', build_preprocessor()),
            ('model', model)
        ])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        results[name] = round(r2_score(y_test, y_pred), 4)
    return results

# 3. Chạy test và tạo bảng kết quả
score_before = evaluate_models(df_before_fe)
score_after = evaluate_models(df_after_fe)

# Đưa vào DataFrame để in ra bảng so sánh
df_compare = pd.DataFrame([score_before, score_after], index=["Trước FE", " Sau FE"]).T

display(df_compare)

,Trước FE,Sau FE
Linear Regression,0.7241,0.7620
Random Forest,0.8032,0.8228
Gradient Boosting,0.8084,0.8175
